#Import Libraries




In [0]:
import pandas as pd
import numpy as np


#Data Ingestion

In [0]:
#customers.csv — 10,000 rows, 5 columns. One row = one customer.
customers= spark.table ('workspace.default._customers')
customers= customers.toPandas()
display(customers.head(5))

CustomerID,Age,City,SignupDate,CustomerSegment
100001,22.0,Tehran,2023-09-11,Regular
100002,55.0,Tabriz,2024-01-16,VIP
100003,49.0,Karaj,2025-07-31,New
100004,39.0,Tehran,2023-04-23,New
100005,38.0,Tehran,2023-04-29,Regular


In [0]:
#orders.csv — 50,120 rows, 8 columns. One row = one order line (one product on one order).
orders= spark.table ('workspace.default._orders')
orders= orders.toPandas()
display(orders.head(5))

OrderID,CustomerID,OrderDate,ProductID,Quantity,Discount,PaymentMethod,Status
500001,103695,2025-08-28,2003,4.0,0.1,Gateway,Completed
500002,107935,2024-05-31,2014,1.0,0.1,Wallet,Completed
500003,105141,2025-08-10,2002,1.0,0.2,CardToCard,Completed
500004,108780,2024-10-11,2012,1.0,0.1,Gateway,Completed
500005,101187,2024-02-19,2008,2.0,0.0,Gateway,Completed


In [0]:
#payments.csv — 50,000 rows, 4 columns. One row = one payment attempt.
payments= spark.table ('workspace.default._payments')
payments= payments.toPandas()
display(payments.head(5))

PaymentID,OrderID,PaymentDate,PaymentStatus
800001,500001,2025-08-28,Paid
800002,500002,2024-05-31,Paid
800003,500003,2025-08-10,Paid
800004,500004,2024-10-11,Paid
800005,500005,2024-02-19,Paid


In [0]:
#products.csv — 20 rows, 4 columns. One row = one product the shop sells.
products= spark.table ('workspace.default._products')
products= products.toPandas()
display(products.head(5))



ProductID,ProductName,Category,UnitPrice
2001,Wireless Mouse,Electronics,18.0
2002,Mechanical Keyboard,Electronics,62.0
2003,USB-C Cable,Accessories,9.0
2004,Power Bank,Electronics,35.0
2005,Laptop Stand,Accessories,28.0


#Exploratory Data Analysis

Step 1: Get to know the data 

customers.csv → Which cities and customer segments (New, Regular, VIP) does the shop have the most customers in?

orders.csv → How many orders come in each month, and is that number growing or shrinking?

payments.csv → What share of payments fail, and does that vary by anything?

products.csv → Which products and categories does the shop actually sell, and at what price?


In [0]:
# products: any blanks, or prices that look wrong? More Insights on product categories: 20 products, 6 categories (top: Electronics with 9 products), no nulls
print(products.isna().sum())
print(products["Category"].describe())
print(products["UnitPrice"].describe())

# payments: any order billed more than once, or pointing to an order that doesn't exist?
print("Duplicate payment OrderIDs:", payments["OrderID"].duplicated().sum())
print("Payments with no matching order:", (~payments["OrderID"].isin(orders["OrderID"])).sum())

# customers: any duplicate customer IDs?
print("Duplicate customer IDs:", customers["CustomerID"].duplicated().sum())

#Orders: On average how many orders do customers place per year?
order_years = pd.to_datetime(orders["OrderDate"]).dt.year
orders_per_customer_year = orders.groupby([order_years, "CustomerID"]).size()
print("Average order by customer per year:", round(orders_per_customer_year.mean(), 2))


ProductID      0
ProductName    0
Category       0
UnitPrice      0
dtype: int64
count              20
unique              6
top       Electronics
freq                9
Name: Category, dtype: object
count     20.000000
mean      61.400000
std       62.836798
min        7.000000
25%       20.250000
50%       45.000000
75%       69.750000
max      260.000000
Name: UnitPrice, dtype: float64
Duplicate payment OrderIDs: 0
Payments with no matching order: 0
Duplicate customer IDs: 0
Average order by customer per year: 2.18


Step 2: Clean the data

Starting with cleaning orders dataset

In [0]:

# 1. Drop exact duplicate rows
before = len(orders)
orders = orders.drop_duplicates()
print(f"Dropped {before - len(orders)} exact duplicate order rows -> {len(orders)} left")

# 2. Drop placeholder/test customer id (breaks the customer join)- Decision: drop those 30 orders, since they can't be tied to a real customer, city, or segment.
before = len(orders)
orders = orders[orders["CustomerID"] != 999999]
print(f"Dropped {before - len(orders)} orders tied to placeholder CustomerID 999999")

# 3. Drop rows missing OrderDate (can't be placed on a monthly trend)
before = len(orders)
orders = orders.dropna(subset=["OrderDate"])
print(f"Dropped {before - len(orders)} orders missing OrderDate")

# 4. Invalid quantities: negative or zero -> data entry errors, drop
before = len(orders)
orders = orders[~(orders["Quantity"].notna() & (orders["Quantity"] <= 0))]
print(f"Dropped {before - len(orders)} orders with quantity <= 0")

# 5. Missing Quantity: can't compute revenue reliably, drop
before = len(orders)
orders = orders.dropna(subset=["Quantity"])
print(f"Dropped {before - len(orders)} orders missing Quantity")

# 6. Missing Discount: assume no discount applied
n_missing_disc = orders["Discount"].isna().sum()
orders["Discount"] = orders["Discount"].fillna(0)
print(f"Filled {n_missing_disc} missing Discount values with 0")

# 7. Missing PaymentMethod- label Unknown, keep the row
n_missing_pm = orders["PaymentMethod"].isna().sum()
orders["PaymentMethod"] = orders["PaymentMethod"].fillna("Unknown")
print(f"Filled {n_missing_pm} missing PaymentMethod values with 'Unknown'")

# 8. Change date from string to datetime
orders["OrderDate"] = pd.to_datetime(orders["OrderDate"])
print(f"CLEAN ORDERS COUNT: {len(orders)}")
display(orders.head(5))

Dropped 120 exact duplicate order rows -> 50000 left
Dropped 30 orders tied to placeholder CustomerID 999999
Dropped 35 orders missing OrderDate
Dropped 25 orders with quantity <= 0
Dropped 80 orders missing Quantity
Filled 220 missing Discount values with 0
Filled 448 missing PaymentMethod values with 'Unknown'
CLEAN ORDERS COUNT: 49830


OrderID,CustomerID,OrderDate,ProductID,Quantity,Discount,PaymentMethod,Status
500001,103695,2025-08-28T00:00:00.000Z,2003,4.0,0.1,Gateway,Completed
500002,107935,2024-05-31T00:00:00.000Z,2014,1.0,0.1,Wallet,Completed
500003,105141,2025-08-10T00:00:00.000Z,2002,1.0,0.2,CardToCard,Completed
500004,108780,2024-10-11T00:00:00.000Z,2012,1.0,0.1,Gateway,Completed
500005,101187,2024-02-19T00:00:00.000Z,2008,2.0,0.0,Gateway,Completed


Cleaning Customers Dataset

In [0]:
# Adjust to whatever variants your data has, both in the City column — same city, different spelling.Decision: replace "tehran" with "Tehran"
city_map = {"tehran": "Tehran", "Mashad": "Mashhad"}  

before_unique = customers["City"].nunique(dropna=True)
customers["City"] = customers["City"].replace(city_map)
print(f"Standardized city spelling: {before_unique} -> {customers['City'].nunique(dropna=True)} unique values")

# Fill with "Unknown" — dropping would lose otherwise-good order data
n_missing_city = customers["City"].isna().sum()
customers["City"] = customers["City"].fillna("Unknown")
print(f"Filled {n_missing_city} missing City values with 'Unknown'")

#Keep as-is — Age isn't used by any of the business questions
n_missing_age = customers["Age"].isna().sum()
print(f"{n_missing_age} blank,keep as-is, age isn't used by any of the business questions")

display(customers.head(5))

Standardized city spelling: 12 -> 10 unique values
Filled 119 missing City values with 'Unknown'
180 blank,keep as-is, age isn't used by any of the business questions


CustomerID,Age,City,SignupDate,CustomerSegment
100001,22.0,Tehran,2023-09-11,Regular
100002,55.0,Tabriz,2024-01-16,VIP
100003,49.0,Karaj,2025-07-31,New
100004,39.0,Tehran,2023-04-23,New
100005,38.0,Tehran,2023-04-29,Regular


Checked payments table for duplicate order IDs and orphaned records — none found, no data cleaning needed

Checked products table for duplicates- none found, no data cleaning needed

Step 3: Join the tables 



In [0]:



# Row counts before joining, so we can sanity-check the row
# count after joining against these
print("orders:   ", len(orders))
print("products: ", len(products))
print("customers:", len(customers))
print("payments: ", len(payments))
 


#1.Join tables: orders link to customers through CustomerID, to products through ProductID, and to payments through OrderID. 

df = orders.merge(products, on="ProductID")
df = df.merge(customers, on="CustomerID")
df = df.merge(payments[["OrderID", "PaymentStatus"]], on="OrderID", how="left")
df["PaymentStatus"] = df["PaymentStatus"].fillna("Unknown")
 
print("Rows after joining:", len(df))
 




orders:    49830
products:  20
customers: 10000
payments:  50000
Rows after joining: 49830


Step 4: Build derived columns

In [0]:
 #Create new columns

# Revenue = quantity x price x (1 - discount), rounded to 2 decimal places
df["Revenue"] = (df["Quantity"] * df["UnitPrice"] * (1 - df["Discount"])).round(2)
 
# Year and month
df["OrderDate"] = pd.to_datetime(df["OrderDate"])
df["Year"] = df["OrderDate"].dt.year
df["Month"] = df["OrderDate"].dt.month
df["YearMonth"] = df["OrderDate"].dt.strftime("%Y-%m")
 
# Only completed AND paid orders count as revenue,the only combination where the shop truly (a) delivered the order and (b) kept the money is Status = Completed AND PaymentStatus = Paid.
good = df[(df["Status"] == "Completed") & (df["PaymentStatus"] == "Paid")]

# Show the new joined table
display(df.head(5))

OrderID,CustomerID,OrderDate,ProductID,Quantity,Discount,PaymentMethod,Status,ProductName,Category,UnitPrice,Age,City,SignupDate,CustomerSegment,PaymentStatus,Revenue,Year,Month,YearMonth
500001,103695,2025-08-28T00:00:00.000Z,2003,4.0,0.1,Gateway,Completed,USB-C Cable,Accessories,9.0,36.0,Qom,2024-03-05,Regular,Paid,32.4,2025,8,2025-08
500002,107935,2024-05-31T00:00:00.000Z,2014,1.0,0.1,Wallet,Completed,Backpack,Accessories,42.0,49.0,Kerman,2025-06-04,Regular,Paid,37.8,2024,5,2024-05
500003,105141,2025-08-10T00:00:00.000Z,2002,1.0,0.2,CardToCard,Completed,Mechanical Keyboard,Electronics,62.0,36.0,Tehran,2025-10-19,Regular,Paid,49.6,2025,8,2025-08
500004,108780,2024-10-11T00:00:00.000Z,2012,1.0,0.1,Gateway,Completed,Office Chair,Home Office,180.0,45.0,Shiraz,2023-12-07,Regular,Paid,162.0,2024,10,2024-10
500005,101187,2024-02-19T00:00:00.000Z,2008,2.0,0.0,Gateway,Completed,Phone Case,Accessories,14.0,24.0,Karaj,2023-12-02,Regular,Paid,28.0,2024,2,2024-02


Step 5: Answer the Business Questions

In [0]:
# Answer the business questions


# ---------------------------------------------------------
# Q1: How much revenue did the shop make, from how many
# orders? What is the average order value?
# ---------------------------------------------------------
total_revenue = good["Revenue"].sum()
total_orders = len(good)
avg_order_value = round(total_revenue / total_orders, 2)
kpi_revenue = pd.DataFrame({"Total Revenue": [round(total_revenue, 2)]})
kpi_orders  = pd.DataFrame({"Total Orders": [total_orders]})
kpi_aov     = pd.DataFrame({"Avg Order Value": [avg_order_value]})

display(kpi_revenue)
display(kpi_orders)
display(kpi_aov)
 
 
# ---------------------------------------------------------
# Q2: Is revenue growing, shrinking or flat month by month?
# Are there any seasonal peaks?
# ---------------------------------------------------------
monthly_revenue = good.groupby("YearMonth")["Revenue"].sum().reset_index()
display(monthly_revenue.head(5))
 
 
# ---------------------------------------------------------
# Q3: Which products and categories bring in the most
# revenue? Which sell the most units? Are these the same?
# ---------------------------------------------------------
category_revenue = good.groupby("Category")["Revenue"].sum().sort_values(ascending=False).reset_index()
display(category_revenue.head(5))
 
top_products_revenue = good.groupby("ProductName")["Revenue"].sum().sort_values(ascending=False).head(5).reset_index()
display(top_products_revenue.head(5))
 
# Units sold by product
top_products_units = good.groupby("ProductName")["Quantity"].sum().sort_values(ascending=False).head(5).reset_index()
display(top_products_units.head(5))
 
 
# ---------------------------------------------------------
# Q4: Which cities and customer segments (New, Regular, VIP)
# are most valuable?
# ---------------------------------------------------------
city_revenue = good.groupby("City")["Revenue"].sum().sort_values(ascending=False).reset_index().round(2)
display(city_revenue.head(5))
 
# Revenue per customer, by segment
segment_revenue = good.groupby("CustomerSegment")["Revenue"].sum()
segment_customers = good.groupby("CustomerSegment")["CustomerID"].nunique()
segment_summary = (segment_revenue / segment_customers).reset_index(name="RevenuePerCustomer").round(2)
display(segment_summary.head(5))
 
 
# ---------------------------------------------------------
# Q5: What share of orders are cancelled or returned? What
# share of payments fail? Do some payment methods fail more
# often?
# ---------------------------------------------------------
status_share = (df["Status"].value_counts(normalize=True) * 100).reset_index().round(2)
status_share.columns = ["Status", "Percent"]
display(status_share.head(5))
 
payment_share = (df["PaymentStatus"].value_counts(normalize=True) * 100).reset_index().round(2)
payment_share.columns = ["PaymentStatus", "Percent"]
display(payment_share.head(5))
 
# Payment failure rate by payment method
df["Failed"] = df["PaymentStatus"] == "Failed"
failure_by_method = (df.groupby("PaymentMethod")["Failed"].mean() * 100).reset_index(name="FailureRatePercent").round(2)
display(failure_by_method.head(5))
 
 
# ---------------------------------------------------------
# Q6: Do bigger discounts lead to bigger orders, or just
# lower revenue?
# ---------------------------------------------------------
discount_effect = good.groupby("Discount")[["Quantity", "Revenue"]].mean().reset_index().round(2)
display(discount_effect.head(5))
 
 
# Step 6 - Visualize the results
# # Summary: all the headline answers in one table
 
 
# One row per question, with a plain-language answer, so it's
# all viewable in a single table instead of scrolling back up
summary = pd.DataFrame({
    "Question": [
        "Q1: Total revenue",
        "Q1: Total orders",
        "Q1: Average order value",
        "Q2: Best month (revenue)",
        "Q3: Top category (revenue)",
        "Q3: Top product (revenue)",
        "Q4: Top city (revenue)",
        "Q4: Best segment (revenue per customer)",
        "Q5: Cancelled + returned orders",
        "Q5: Payment failure rate",
        "Q6: Revenue at 0% discount vs. highest discount",
    ],
    "Answer": [
        round(total_revenue, 2),
        total_orders,
        avg_order_value,
        monthly_revenue.loc[monthly_revenue["Revenue"].idxmax(), "YearMonth"],
        category_revenue.iloc[0]["Category"],
        top_products_revenue.iloc[0]["ProductName"],
        city_revenue.iloc[0]["City"],
        segment_summary.loc[segment_summary["RevenuePerCustomer"].idxmax(), "CustomerSegment"],
        f'{status_share[status_share["Status"].isin(["Cancelled", "Returned"])]["Percent"].sum():.2f}%',
        f'{payment_share[payment_share["PaymentStatus"] == "Failed"]["Percent"].iloc[0]:.2f}%',
        f'{discount_effect["Revenue"].iloc[0]:.2f} vs. {discount_effect["Revenue"].iloc[-1]:.2f}',
    ],
})
summary["Answer"] = summary["Answer"].astype(str)
display(summary)
 
 
## 5. Save the clean table
 
 
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.shop_analysis")

spark.createDataFrame(df).write.mode("overwrite").saveAsTable("workspace.shop_analysis.clean_orders")
 
 



Total Revenue
2988379.8


Databricks visualization. Run in Databricks to view.

Total Orders
42696


Databricks visualization. Run in Databricks to view.

Avg Order Value
69.99


...

PaymentMethod,FailureRatePercent
CardToCard,3.9
Cash,3.65
Gateway,3.87
Unknown,4.02
Wallet,3.87


Databricks visualization. Run in Databricks to view.

Discount,Quantity,Revenue
0.0,1.87,74.59
0.05,1.9,71.59
0.1,1.89,68.94
0.15,1.88,65.0
0.2,1.87,61.36


Databricks visualization. Run in Databricks to view.

Question,Answer
Q1: Total revenue,2988379.8
Q1: Total orders,42696
Q1: Average order value,69.99
Q2: Best month (revenue),2024-03
Q3: Top category (revenue),Electronics
Q3: Top product (revenue),Headphones
Q4: Top city (revenue),Tehran
Q4: Best segment (revenue per customer),New
Q5: Cancelled + returned orders,8.01%
Q5: Payment failure rate,3.86%
